# Test Statistical Methods By TODS Anomaly Type

This notebook runs one selected statistical method on TODS point-anomaly datasets and on TODS sequence-anomaly datasets in two separate cells.


In [6]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.model_wrapper import run_Unsupervise_AD

warnings.filterwarnings("ignore")


## Configuration


In [7]:
METHOD_NAME = "POLY"  # "Sub_PCA", "POLY", "KMeansAD", "KShapeAD"

SUPPORTED_METHODS = ["Sub_PCA", "POLY", "KMeansAD", "KShapeAD"]
POINT_DATASET_IDS = [1, 3, 5, 6, 10]
SEQUENCE_DATASET_IDS = [2, 4, 8, 9, 11, 12, 13, 14, 15]

DATASET_DIRECTORY = Path("../../datasets/TSB-AD-U")
RESULTS_DIRECTORY = Path("../../results/TODS/by_anomaly_type")
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

if METHOD_NAME not in SUPPORTED_METHODS:
    raise ValueError(f"Unsupported METHOD_NAME: {METHOD_NAME}. Choose from {SUPPORTED_METHODS}")

print(f"Method: {METHOD_NAME}")
print(f"Dataset directory: {DATASET_DIRECTORY}")
print(f"Results directory: {RESULTS_DIRECTORY}")


Method: POLY
Dataset directory: ../../datasets/TSB-AD-U
Results directory: ../../results/TODS/by_anomaly_type


In [ ]:
def get_dataset_paths(dataset_ids):
    dataset_paths = []

    for dataset_id in dataset_ids:
        pattern = f"*TODS_id_{dataset_id}_*.csv"
        matches = sorted(DATASET_DIRECTORY.glob(pattern), key=lambda path: path.name)

        if len(matches) == 0:
            print(f"No dataset found for TODS id {dataset_id} with pattern {pattern}")
            continue

        if len(matches) > 1:
            print(f"Multiple datasets found for TODS id {dataset_id}: {[path.name for path in matches]}")
            continue

        dataset_paths.append(matches[0])

    return dataset_paths


def evaluate_dataset_group(group_name, group_key, dataset_ids):
    dataset_paths = get_dataset_paths(dataset_ids)

    print(f"\n{group_name} datasets: {[path.name for path in dataset_paths]}")

    rows = []

    for dataset_path in dataset_paths:

        df = pd.read_csv(dataset_path).dropna()
        data = df.iloc[:, :-1].values.astype(float)
        labels = df["Label"].astype(int).to_numpy()

        scores = run_Unsupervise_AD(METHOD_NAME, data)

        if isinstance(scores, str):
            print(f"Error running {METHOD_NAME} on {dataset_path.name}: {scores}")
            continue

        scores = np.asarray(scores).ravel()

        if len(scores) != len(labels):
            print(f"Length mismatch for {dataset_path.name}: scores={len(scores)}, labels={len(labels)}")

        n = min(len(scores), len(labels))
        scores = scores[:n]
        labels = labels[:n]

        metrics = get_metrics(scores, labels)

        row = {
            "dataset": dataset_path.name,
            "method": METHOD_NAME,
        }

        for metric_name, metric_value in metrics.items():
            row[metric_name] = float(metric_value)

        rows.append(row)

    results_df = pd.DataFrame(rows)
    display(results_df)

    results_path = RESULTS_DIRECTORY / f"results_{METHOD_NAME}_TODS_{group_key}.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved results to: {results_path}")

    if results_df.empty:
        print(f"No results available for {group_name}.")
        return results_df

    print(f"\nMean metrics for {group_name}:")
    for metric_name in results_df.columns[2:]:
        print(f"Mean {metric_name}: {results_df[metric_name].mean():.4f}")

    return results_df


## Point-Anomaly TODS Datasets


In [9]:
point_results_df = evaluate_dataset_group("point-anomaly TODS", "point", POINT_DATASET_IDS)



point-anomaly TODS datasets: ['287_TODS_id_1_Synthetic_tr_500_1st_11.csv', '289_TODS_id_3_Synthetic_tr_500_1st_26.csv', '291_TODS_id_5_Synthetic_tr_500_1st_11.csv', '292_TODS_id_6_Synthetic_tr_500_1st_11.csv', '296_TODS_id_10_Synthetic_tr_500_1st_26.csv']
--- Processing 287_TODS_id_1_Synthetic_tr_500_1st_11.csv ---
--- Processing 289_TODS_id_3_Synthetic_tr_500_1st_26.csv ---
--- Processing 291_TODS_id_5_Synthetic_tr_500_1st_11.csv ---
--- Processing 292_TODS_id_6_Synthetic_tr_500_1st_11.csv ---
--- Processing 296_TODS_id_10_Synthetic_tr_500_1st_26.csv ---


,dataset,method,AUC-PR,AUC-ROC,VUS-PR,VUS-ROC,Standard-F1,PA-F1,Event-based-F1,R-based-F1,Affiliation-F
0,287_TODS_id_1_Synthetic_tr_500_1st_11.csv,POLY,0.071536,0.590429,0.831184,0.898802,0.136661,0.135524,0.135344,0.021793,0.646305
1,289_TODS_id_3_Synthetic_tr_500_1st_26.csv,POLY,0.062579,0.537464,0.561791,0.766401,0.169693,0.169697,0.119250,0.071839,0.642222
2,291_TODS_id_5_Synthetic_tr_500_1st_11.csv,POLY,0.042590,0.419273,0.722741,0.862903,0.099220,0.098719,0.098709,0.019121,0.645960
3,292_TODS_id_6_Synthetic_tr_500_1st_11.csv,POLY,0.056378,0.518959,0.800714,0.883414,0.110162,0.109437,0.108617,0.019052,0.645960
4,296_TODS_id_10_Synthetic_tr_500_1st_26.csv,POLY,0.067581,0.559690,0.682319,0.825329,0.162247,0.162252,0.108333,0.049495,0.638378


Saved results to: ../../results/TODS/by_anomaly_type/results_POLY_TODS_point.csv

Mean metrics for point-anomaly TODS:
Mean AUC-PR: 0.0601
Mean AUC-ROC: 0.5252
Mean VUS-PR: 0.7197
Mean VUS-ROC: 0.8474
Mean Standard-F1: 0.1356
Mean PA-F1: 0.1351
Mean Event-based-F1: 0.1141
Mean R-based-F1: 0.0363
Mean Affiliation-F: 0.6438


## Sequence-Anomaly TODS Datasets


In [10]:
sequence_results_df = evaluate_dataset_group("sequence-anomaly TODS", "sequence", SEQUENCE_DATASET_IDS)



sequence-anomaly TODS datasets: ['288_TODS_id_2_Synthetic_tr_500_1st_65.csv', '290_TODS_id_4_Synthetic_tr_500_1st_26.csv', '294_TODS_id_8_Synthetic_tr_500_1st_200.csv', '295_TODS_id_9_Synthetic_tr_1250_1st_2046.csv', '297_TODS_id_11_Synthetic_tr_500_1st_7.csv', '298_TODS_id_12_Synthetic_tr_500_1st_0.csv', '299_TODS_id_13_Synthetic_tr_500_1st_65.csv', '300_TODS_id_14_Synthetic_tr_1250_1st_2555.csv', '301_TODS_id_15_Synthetic_tr_500_1st_245.csv']
--- Processing 288_TODS_id_2_Synthetic_tr_500_1st_65.csv ---
--- Processing 290_TODS_id_4_Synthetic_tr_500_1st_26.csv ---
--- Processing 294_TODS_id_8_Synthetic_tr_500_1st_200.csv ---
--- Processing 295_TODS_id_9_Synthetic_tr_1250_1st_2046.csv ---
--- Processing 297_TODS_id_11_Synthetic_tr_500_1st_7.csv ---
--- Processing 298_TODS_id_12_Synthetic_tr_500_1st_0.csv ---
--- Processing 299_TODS_id_13_Synthetic_tr_500_1st_65.csv ---
--- Processing 300_TODS_id_14_Synthetic_tr_1250_1st_2555.csv ---
--- Processing 301_TODS_id_15_Synthetic_tr_500_1st_24

,dataset,method,AUC-PR,AUC-ROC,VUS-PR,VUS-ROC,Standard-F1,PA-F1,Event-based-F1,R-based-F1,Affiliation-F
0,288_TODS_id_2_Synthetic_tr_500_1st_65.csv,POLY,0.379792,0.797445,0.669975,0.804410,0.576798,0.617960,0.245545,0.090431,0.639069
1,290_TODS_id_4_Synthetic_tr_500_1st_26.csv,POLY,0.355365,0.805284,0.518498,0.692979,0.541899,0.531915,0.193743,0.200380,0.637891
2,294_TODS_id_8_Synthetic_tr_500_1st_200.csv,POLY,0.466746,0.721058,0.482108,0.683178,0.542852,0.553250,0.431793,0.349051,0.758902
3,295_TODS_id_9_Synthetic_tr_1250_1st_2046.csv,POLY,0.226823,0.731849,0.362609,0.720689,0.402701,0.490928,0.350189,0.306155,0.696449
4,297_TODS_id_11_Synthetic_tr_500_1st_7.csv,POLY,0.106056,0.658095,0.436920,0.730518,0.185001,0.196379,0.185956,0.144928,0.726744
5,298_TODS_id_12_Synthetic_tr_500_1st_0.csv,POLY,0.222081,0.779600,0.561922,0.750270,0.454541,0.485684,0.272727,0.159902,0.674013
6,299_TODS_id_13_Synthetic_tr_500_1st_65.csv,POLY,0.082568,0.610237,0.685974,0.833319,0.165539,0.158171,0.156162,0.051272,0.653713
7,300_TODS_id_14_Synthetic_tr_1250_1st_2555.csv,POLY,0.096351,0.411304,0.122215,0.493153,0.166665,0.529801,0.268657,0.162085,0.709863
8,301_TODS_id_15_Synthetic_tr_500_1st_245.csv,POLY,0.077253,0.400967,0.215547,0.539344,0.166111,0.166113,0.165312,0.076046,0.678787


Saved results to: ../../results/TODS/by_anomaly_type/results_POLY_TODS_sequence.csv

Mean metrics for sequence-anomaly TODS:
Mean AUC-PR: 0.2237
Mean AUC-ROC: 0.6573
Mean VUS-PR: 0.4506
Mean VUS-ROC: 0.6942
Mean Standard-F1: 0.3558
Mean PA-F1: 0.4145
Mean Event-based-F1: 0.2522
Mean R-based-F1: 0.1711
Mean Affiliation-F: 0.6862
